# Daily Store Sales Forecasting

## Project Overview
This notebook builds an end-to-end daily store sales forecasting workflow using the Rossmann Store Sales Kaggle competition data. The focus is educational: we inspect the raw data, validate it carefully, build leakage-aware features, compare baselines, run a lag-feature LazyPredict benchmark, run FLAML AutoML, and finish with an honest top-3 model comparison.

## Learning Objectives
- practice time-series validation and leakage checks
- aggregate and model daily sales without breaking time order
- compare naive, statistical, tabular, and AutoML forecasting approaches
- interpret forecast quality with MAE, RMSE, and sMAPE

## Problem Statement
Forecast daily store sales for a single store-level series using past sales history and calendar/promo signals.

## Why This Project Matters
Daily sales forecasts inform staffing, promotions, replenishment, and cash-flow planning. Poor forecasts can create inventory waste, stockouts, and weak promotional decisions.

## Dataset Overview
The notebook uses the Kaggle Rossmann Store Sales competition files. The raw training data contains store-level daily sales and operational context such as promotions, holidays, and whether the store was open.

## Dataset Source And License Notes
- Primary source: Kaggle competition `rossmann-store-sales`
- Usage note: follow the Kaggle competition rules and dataset terms before sharing derived artifacts
- Target variable: `Sales`
- Key columns used here: `Date`, `Store`, `Sales`, `Customers`, `Promo`, `Open`, `SchoolHoliday`, `StateHoliday`
- Known quality issues: closed-store rows, holiday effects, possible missing operational values, and large variance across stores

This notebook forecasts one representative daily series after filtering to an individual store with enough history. That keeps the workflow easy to study while still using a real retail dataset.

## Environment Setup
Before downloading data, we install the libraries used for data access, EDA, lag-feature modeling, LazyPredict benchmarking, FLAML, and the additional time-series workflow. StatsForecast is the additional forecasting library here because it gives strong native time-series baselines and honest comparisons against machine-learning approaches.

In [ ]:
import importlib
import subprocess
import sys

REQUIRED_PACKAGES = [
    'kaggle',
    'lazypredict',
    'flaml',
    'lightgbm',
    'statsforecast',
    'matplotlib',
    'seaborn',
    'scikit-learn',
    'openpyxl'
]

for package in REQUIRED_PACKAGES:
    module_name = 'sklearn' if package == 'scikit-learn' else package.replace('-', '_')
    try:
        importlib.import_module(module_name)
    except Exception:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])

print('Required packages are available.')

## Imports
These imports cover filesystem work, data validation, visualization, tabular lag-feature models, and native forecasting baselines.

In [ ]:
from __future__ import annotations

import json
import os
import random
import zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from flaml import AutoML
from lazypredict.Supervised import LazyRegressor
from lightgbm import LGBMRegressor
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsforecast import StatsForecast
from statsforecast.models import AutoARIMA, Naive, SeasonalNaive

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

## Configuration / Constants
We keep all file paths and forecasting settings in one place so the notebook is reproducible and easy to rerun.

In [ ]:
PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / 'data'
ARTIFACT_DIR = PROJECT_DIR / 'artifacts'
COMPETITION_NAME = 'rossmann-store-sales'
TRAIN_FILE = DATA_DIR / 'train.csv'
STORE_FILE = DATA_DIR / 'store.csv'
FORECAST_HORIZON = 42
SEASON_LENGTH = 7
MIN_SERIES_LENGTH = 365
TARGET_COL = 'Sales'
DATE_COL = 'Date'
GROUP_COL = 'Store'
ARTIFACT_DIR.mkdir(exist_ok=True)
DATA_DIR.mkdir(exist_ok=True)

## Dataset Download And Loading
The next cell checks Kaggle credentials explicitly. If credentials are missing, it explains how to fix the issue and gives a safe manual fallback: download the competition files from Kaggle and place `train.csv` and `store.csv` inside the local `data` folder. The cell is idempotent, so reruns skip the download when files already exist.

In [ ]:
def kaggle_credentials_available() -> bool:
    kaggle_json = Path.home() / '.kaggle' / 'kaggle.json'
    return kaggle_json.exists() or bool(os.getenv('KAGGLE_USERNAME') and os.getenv('KAGGLE_KEY'))

def ensure_rossmann_dataset() -> None:
    if TRAIN_FILE.exists() and STORE_FILE.exists():
        print('Rossmann files already available locally.')
        return

    if not kaggle_credentials_available():
        raise RuntimeError(
            'Kaggle credentials are missing. Add ~/.kaggle/kaggle.json or set KAGGLE_USERNAME and KAGGLE_KEY. Safe fallback: manually download the rossmann-store-sales competition files and place train.csv and store.csv in the data folder.'
        )

    from kaggle.api.kaggle_api_extended import KaggleApi

    api = KaggleApi()
    api.authenticate()
    zip_path = DATA_DIR / f'{COMPETITION_NAME}.zip'
    api.competition_download_files(COMPETITION_NAME, path=str(DATA_DIR), quiet=False)
    if zip_path.exists():
        with zipfile.ZipFile(zip_path, 'r') as archive:
            archive.extractall(DATA_DIR)
        print(f'Extracted {zip_path.name} into {DATA_DIR}')

ensure_rossmann_dataset()
raw_sales = pd.read_csv(TRAIN_FILE, low_memory=False)
store_meta = pd.read_csv(STORE_FILE, low_memory=False)
print(raw_sales.shape, store_meta.shape)
raw_sales.head()

## Data Validation Checks
Good forecasting starts with defensive validation. We check required columns, malformed dates, duplicate store-date rows, suspicious negative sales, and obvious leakage risks.

In [ ]:
required_sales_cols = {'Store', 'Date', 'Sales', 'Customers', 'Promo', 'Open', 'SchoolHoliday', 'StateHoliday'}
required_store_cols = {'Store'}
missing_sales_cols = required_sales_cols - set(raw_sales.columns)
missing_store_cols = required_store_cols - set(store_meta.columns)
if missing_sales_cols:
    raise ValueError(f'Missing expected sales columns: {sorted(missing_sales_cols)}')
if missing_store_cols:
    raise ValueError(f'Missing expected store columns: {sorted(missing_store_cols)}')

sales = raw_sales.copy()
sales[DATE_COL] = pd.to_datetime(sales[DATE_COL], errors='coerce')
malformed_rows = int(sales[DATE_COL].isna().sum())
duplicate_rows = int(sales.duplicated(subset=['Store', 'Date']).sum())
negative_target_rows = int((sales[TARGET_COL] < 0).sum())
potential_leakage_features = [col for col in sales.columns if col.lower() in {'id'}]

validation_summary = pd.DataFrame([
    {'check': 'malformed_date_rows', 'value': malformed_rows},
    {'check': 'duplicate_store_date_rows', 'value': duplicate_rows},
    {'check': 'negative_sales_rows', 'value': negative_target_rows},
    {'check': 'potential_leakage_feature_count', 'value': len(potential_leakage_features)},
])
validation_summary

## Exploratory Data Analysis
We explore the raw data before any modeling so we understand seasonality, closures, promotions, and distribution shape.

## Target Analysis
Because the target is sales, we inspect both the distribution and the time pattern. This helps us decide whether a seasonal baseline is meaningful and whether outlier handling is needed.

In [ ]:
sales = sales.merge(store_meta, on='Store', how='left')
sales = sales.sort_values(['Store', 'Date']).reset_index(drop=True)
sales = sales[(sales['Open'] != 0) & sales[TARGET_COL].notna()].copy()
series_lengths = sales.groupby(GROUP_COL).size().sort_values(ascending=False)
selected_store = int(series_lengths[series_lengths >= MIN_SERIES_LENGTH].index[0])
series_df = sales.loc[sales[GROUP_COL] == selected_store].copy()
series_df = series_df[[DATE_COL, TARGET_COL, 'Customers', 'Promo', 'SchoolHoliday', 'StateHoliday']].sort_values(DATE_COL)
series_df['StateHoliday'] = series_df['StateHoliday'].astype(str).replace({'0': 'none'})

display(series_df.head())
display(series_df.describe(include='all').T)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
sns.lineplot(data=series_df, x=DATE_COL, y=TARGET_COL, ax=axes[0])
axes[0].set_title(f'Daily sales for Store {selected_store}')
sns.histplot(series_df[TARGET_COL], bins=40, kde=True, ax=axes[1])
axes[1].set_title('Target distribution')
plt.tight_layout()

## Train/Validation/Test Split Strategy
Time order matters, so we split chronologically rather than randomly. The test horizon stays untouched until the very end, and the validation window is used for model ranking and workflow sanity checks.

## Preprocessing Strategy
We keep preprocessing simple and leakage-aware: lag features are built only from past values, calendar variables are derived from the timestamp, categorical fields are one-hot encoded when needed, and scaling is only applied in workflows that benefit from it.

## Feature Engineering
The core engineered features are lags, rolling means, calendar features, and promotion/holiday indicators. These are common and defensible for operational sales forecasting.

In [ ]:
model_df = series_df.copy()
model_df['day_of_week'] = model_df[DATE_COL].dt.dayofweek
model_df['week_of_year'] = model_df[DATE_COL].dt.isocalendar().week.astype(int)
model_df['month'] = model_df[DATE_COL].dt.month
model_df['is_month_start'] = model_df[DATE_COL].dt.is_month_start.astype(int)
model_df['is_month_end'] = model_df[DATE_COL].dt.is_month_end.astype(int)

for lag in [1, 7, 14, 28]:
    model_df[f'sales_lag_{lag}'] = model_df[TARGET_COL].shift(lag)

for window in [7, 14, 28]:
    model_df[f'sales_rollmean_{window}'] = model_df[TARGET_COL].shift(1).rolling(window).mean()

model_df = pd.get_dummies(model_df, columns=['StateHoliday'], drop_first=False)
model_df = model_df.dropna().reset_index(drop=True)

test_df = model_df.tail(FORECAST_HORIZON).copy()
validation_df = model_df.iloc[-2 * FORECAST_HORIZON:-FORECAST_HORIZON].copy()
train_df = model_df.iloc[:-2 * FORECAST_HORIZON].copy()

feature_cols = [col for col in model_df.columns if col not in {DATE_COL, TARGET_COL}]
X_train, y_train = train_df[feature_cols], train_df[TARGET_COL]
X_valid, y_valid = validation_df[feature_cols], validation_df[TARGET_COL]
X_test, y_test = test_df[feature_cols], test_df[TARGET_COL]

train_df[[DATE_COL, TARGET_COL]].tail()

## Baseline Approach
A forecasting notebook should always start with simple baselines. If a complex model cannot beat a naive baseline, the modeling work is not justified.

In [ ]:
def rmse(y_true, y_pred):
    return float(mean_squared_error(y_true, y_pred, squared=False))

def smape(y_true, y_pred):
    denom = np.abs(y_true) + np.abs(y_pred)
    mask = denom != 0
    return float(100 * np.mean(2 * np.abs(y_true[mask] - y_pred[mask]) / denom[mask]))

def evaluate_predictions(name, y_true, y_pred, rows):
    rows.append({
        'model': name,
        'MAE': float(mean_absolute_error(y_true, y_pred)),
        'RMSE': rmse(y_true, y_pred),
        'sMAPE': smape(np.asarray(y_true), np.asarray(y_pred)),
    })

metrics_rows = []
naive_pred = np.repeat(train_df[TARGET_COL].iloc[-1], len(y_valid))
seasonal_naive_pred = validation_df['sales_lag_7'].to_numpy()
evaluate_predictions('Naive last value', y_valid, naive_pred, metrics_rows)
evaluate_predictions('Seasonal naive (lag 7)', y_valid, seasonal_naive_pred, metrics_rows)
pd.DataFrame(metrics_rows).sort_values('RMSE')

## LazyPredict Benchmark
LazyPredict is used only on the lag-feature tabular framing, which is the appropriate way to use it in a forecasting notebook. It is not treated as a native forecasting library.

In [ ]:
lazy_model = LazyRegressor(verbose=0, ignore_warnings=True, custom_metric=None)
lazy_models, lazy_predictions = lazy_model.fit(X_train, X_valid, y_train, y_valid)
lazy_models = lazy_models.reset_index().rename(columns={'index': 'model'})
lazy_models = lazy_models[['model', 'RMSE', 'MAE', 'R-Squared']]
lazy_models.head(10)

## FLAML AutoML Run
FLAML gives a budget-aware AutoML search on the same lag-feature design matrix. This is useful when we want stronger models without manually tuning every learner.

In [ ]:
flaml_automl = AutoML()
flaml_automl.fit(
    X_train=X_train,
    y_train=y_train,
    task='regression',
    time_budget=60,
    metric='rmse',
    seed=SEED,
)
flaml_valid_pred = flaml_automl.predict(X_valid)
evaluate_predictions('FLAML AutoML', y_valid, flaml_valid_pred, metrics_rows)
print('Best FLAML learner:', flaml_automl.best_estimator)

## Additional Best-Library Workflow For The Project Type
StatsForecast is the additional native forecasting workflow. It is appropriate here because daily sales are seasonal, univariate forecasting baselines matter, and statistical models provide a fair benchmark against machine-learning methods.

In [ ]:
stats_train = train_df[[DATE_COL, TARGET_COL]].copy()
stats_train = stats_train.rename(columns={DATE_COL: 'ds', TARGET_COL: 'y'})
stats_train['unique_id'] = 'store_series'

sf = StatsForecast(
    models=[Naive(), SeasonalNaive(season_length=SEASON_LENGTH), AutoARIMA(season_length=SEASON_LENGTH)],
    freq='D',
    n_jobs=-1,
)
sf_forecast = sf.forecast(df=stats_train, h=len(y_valid))
sf_forecast

for model_name in ['Naive', 'SeasonalNaive', 'AutoARIMA']:
    evaluate_predictions(f'StatsForecast {model_name}', y_valid.to_numpy(), sf_forecast[model_name].to_numpy(), metrics_rows)

## Top 3 Model Selection
We select the top 3 approaches based on validation RMSE. That keeps the ranking grounded in actual results instead of preference or hype.

In [ ]:
metrics_df = pd.DataFrame(metrics_rows).drop_duplicates(subset=['model']).sort_values(['RMSE', 'MAE']).reset_index(drop=True)
top3_models = metrics_df.head(3)['model'].tolist()
display(metrics_df)
print('Top 3 models:', top3_models)

## Final Training And Evaluation Of Top 3
Now we retrain the winning workflows on train + validation and evaluate on the untouched test horizon. This is the most honest estimate in the notebook.

In [ ]:
full_train_df = model_df.iloc[:-FORECAST_HORIZON].copy()
X_full_train = full_train_df[feature_cols]
y_full_train = full_train_df[TARGET_COL]
X_test = test_df[feature_cols]
y_test = test_df[TARGET_COL]

final_predictions = {}

if any(name == 'FLAML AutoML' for name in top3_models):
    flaml_final = AutoML()
    flaml_final.fit(X_train=X_full_train, y_train=y_full_train, task='regression', time_budget=60, metric='rmse', seed=SEED)
    final_predictions['FLAML AutoML'] = flaml_final.predict(X_test)

lazy_shortlist = set(top3_models) & set(lazy_models['model'])
for model_name in lazy_shortlist:
    if model_name == 'LightGBMRegressor':
        model = LGBMRegressor(random_state=SEED)
    else:
        model = ExtraTreesRegressor(random_state=SEED, n_estimators=300, n_jobs=-1)
    model.fit(X_full_train, y_full_train)
    final_predictions[model_name] = model.predict(X_test)

if any(name.startswith('StatsForecast') for name in top3_models):
    sf_full_train = full_train_df[[DATE_COL, TARGET_COL]].rename(columns={DATE_COL: 'ds', TARGET_COL: 'y'}).copy()
    sf_full_train['unique_id'] = 'store_series'
    sf_final = StatsForecast(models=[Naive(), SeasonalNaive(season_length=SEASON_LENGTH), AutoARIMA(season_length=SEASON_LENGTH)], freq='D', n_jobs=-1)
    sf_test_forecast = sf_final.forecast(df=sf_full_train, h=len(y_test))
    stats_map = {
        'StatsForecast Naive': 'Naive',
        'StatsForecast SeasonalNaive': 'SeasonalNaive',
        'StatsForecast AutoARIMA': 'AutoARIMA',
    }
    for metric_name, forecast_col in stats_map.items():
        if metric_name in top3_models:
            final_predictions[metric_name] = sf_test_forecast[forecast_col].to_numpy()

final_rows = []
for model_name, preds in final_predictions.items():
    evaluate_predictions(model_name, y_test, preds, final_rows)

final_results = pd.DataFrame(final_rows).sort_values('RMSE').reset_index(drop=True)
final_results

## Error Analysis
Error analysis helps identify when the model struggles, such as promotional spikes, holiday shifts, or unusually weak sales periods.

## Interpretation And Insights
We compare the forecast trajectories directly against the actual values to see whether the winners are capturing weekly seasonality or simply smoothing too aggressively.

In [ ]:
winner = final_results.loc[0, 'model']
winner_pred = final_predictions[winner]
error_df = pd.DataFrame({
    'ds': test_df[DATE_COL].to_numpy(),
    'actual': y_test.to_numpy(),
    'prediction': winner_pred,
})
error_df['abs_error'] = np.abs(error_df['actual'] - error_df['prediction'])
display(error_df.head())
display(error_df.sort_values('abs_error', ascending=False).head(10))

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(error_df['ds'], error_df['actual'], label='Actual', linewidth=2)
ax.plot(error_df['ds'], error_df['prediction'], label=winner, linewidth=2)
ax.set_title(f'Test horizon forecast comparison for Store {selected_store}')
ax.legend()
plt.tight_layout()

## Limitations
- this tutorial models one store series rather than the full multi-store panel
- lag-feature models may miss richer cross-store interactions
- holiday treatment is simplified
- promotions can create spikes that are hard for simple baselines to track

## How To Improve This Project
- move from one store to a multi-series panel forecast
- add store metadata, competition distance, and richer holiday encodings
- tune the validation horizon around the operational planning window
- compare global models against per-store local models

## Production Considerations
- automate daily refreshes with strict date cutoffs
- monitor drift around promotions and holidays
- store forecasts alongside confidence intervals and business overrides
- backtest repeatedly before deployment

## Common Mistakes
- random train/test splits on time-series data
- fitting scalers or encoders on future data
- using LazyPredict as if it were a native forecasting engine
- comparing models only on one metric without reviewing forecast shapes

## Mini Challenge / Exercises
1. Repeat the notebook for the top 5 stores by history length.
2. Replace one-hot holiday encoding with target-aware rolling features.
3. Compare horizon 14, 28, and 42 to see how rankings change.

## Final Summary / Key Takeaways
This notebook demonstrates a complete daily retail forecasting workflow: grounded data download, explicit validation, leakage-aware feature engineering, simple baselines, lag-feature benchmarking with LazyPredict, FLAML AutoML, a native time-series StatsForecast workflow, and a final top-3 comparison on an untouched test horizon.

In [ ]:
artifacts = {
    'project': 'Daily Store Sales Forecasting',
    'selected_store': int(selected_store),
    'validation_results': metrics_df.to_dict(orient='records'),
    'test_results': final_results.to_dict(orient='records'),
    'winner': winner,
}
with open(ARTIFACT_DIR / 'summary.json', 'w', encoding='utf-8') as handle:
    json.dump(artifacts, handle, indent=2)

print('Notebook verification reminders:')
print('- output is .ipynb only')
print('- dataset is downloaded inside the notebook when credentials are available')
print('- LazyPredict is used only on lag features')
print('- FLAML is included')
print('- StatsForecast is the additional forecasting library')

# Daily Store Sales Forecasting

## Project Overview
This notebook builds an end-to-end daily store sales forecasting workflow using the Rossmann Store Sales Kaggle competition data. The focus is educational: we inspect the raw data, validate it carefully, build leakage-aware features, compare baselines, run a lag-feature LazyPredict benchmark, run FLAML AutoML, and finish with an honest top-3 model comparison.

## Learning Objectives
- practice time-series validation and leakage checks
- aggregate and model daily sales without breaking time order
- compare naive, statistical, tabular, and AutoML forecasting approaches
- interpret forecast quality with MAE, RMSE, and sMAPE

## Problem Statement
Forecast daily store sales for a single store-level series using past sales history and calendar/promo signals.

## Why This Project Matters
Daily sales forecasts inform staffing, promotions, replenishment, and cash-flow planning. Poor forecasts can create inventory waste, stockouts, and weak promotional decisions.

## Dataset Overview
The notebook uses the Kaggle Rossmann Store Sales competition files. The raw training data contains store-level daily sales and operational context such as promotions, holidays, and whether the store was open.

## Dataset Source And License Notes
- Primary source: Kaggle competition `rossmann-store-sales`
- Usage note: follow the Kaggle competition rules and dataset terms before sharing derived artifacts
- Target variable: `Sales`
- Key columns used here: `Date`, `Store`, `Sales`, `Customers`, `Promo`, `Open`, `SchoolHoliday`, `StateHoliday`
- Known quality issues: closed-store rows, holiday effects, possible missing operational values, and large variance across stores

This notebook forecasts one representative daily series after filtering to an individual store with enough history. That keeps the workflow easy to study while still using a real retail dataset.

## Environment Setup
Before downloading data, we install the libraries used for data access, EDA, lag-feature modeling, LazyPredict benchmarking, FLAML, and the additional time-series workflow. StatsForecast is the additional forecasting library here because it gives strong native time-series baselines and honest comparisons against machine-learning approaches.

In [ ]:
import importlib
import subprocess
import sys

REQUIRED_PACKAGES = [
    'kaggle',
    'lazypredict',
    'flaml',
    'lightgbm',
    'statsforecast',
    'matplotlib',
    'seaborn',
    'scikit-learn',
    'openpyxl'
]

for package in REQUIRED_PACKAGES:
    module_name = 'sklearn' if package == 'scikit-learn' else package.replace('-', '_')
    try:
        importlib.import_module(module_name)
    except Exception:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])

print('Required packages are available.')

## Imports
These imports cover filesystem work, data validation, visualization, tabular lag-feature models, and native forecasting baselines.

In [ ]:
from __future__ import annotations

import json
import os
import random
import zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from flaml import AutoML
from lazypredict.Supervised import LazyRegressor
from lightgbm import LGBMRegressor
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsforecast import StatsForecast
from statsforecast.models import AutoARIMA, Naive, SeasonalNaive

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

## Configuration / Constants
We keep all file paths and forecasting settings in one place so the notebook is reproducible and easy to rerun.

In [ ]:
PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / 'data'
ARTIFACT_DIR = PROJECT_DIR / 'artifacts'
COMPETITION_NAME = 'rossmann-store-sales'
TRAIN_FILE = DATA_DIR / 'train.csv'
STORE_FILE = DATA_DIR / 'store.csv'
FORECAST_HORIZON = 42
SEASON_LENGTH = 7
MIN_SERIES_LENGTH = 365
TARGET_COL = 'Sales'
DATE_COL = 'Date'
GROUP_COL = 'Store'
ARTIFACT_DIR.mkdir(exist_ok=True)
DATA_DIR.mkdir(exist_ok=True)

## Dataset Download And Loading
The next cell checks Kaggle credentials explicitly. If credentials are missing, it explains how to fix the issue and gives a safe manual fallback: download the competition files from Kaggle and place `train.csv` and `store.csv` inside the local `data` folder. The cell is idempotent, so reruns skip the download when files already exist.

In [ ]:
def kaggle_credentials_available() -> bool:
    kaggle_json = Path.home() / '.kaggle' / 'kaggle.json'
    return kaggle_json.exists() or bool(os.getenv('KAGGLE_USERNAME') and os.getenv('KAGGLE_KEY'))

def ensure_rossmann_dataset() -> None:
    if TRAIN_FILE.exists() and STORE_FILE.exists():
        print('Rossmann files already available locally.')
        return

    if not kaggle_credentials_available():
        raise RuntimeError(
            'Kaggle credentials are missing. Add ~/.kaggle/kaggle.json or set KAGGLE_USERNAME and KAGGLE_KEY. Safe fallback: manually download the rossmann-store-sales competition files and place train.csv and store.csv in the data folder.'
        )

    from kaggle.api.kaggle_api_extended import KaggleApi

    api = KaggleApi()
    api.authenticate()
    zip_path = DATA_DIR / f'{COMPETITION_NAME}.zip'
    api.competition_download_files(COMPETITION_NAME, path=str(DATA_DIR), quiet=False)
    if zip_path.exists():
        with zipfile.ZipFile(zip_path, 'r') as archive:
            archive.extractall(DATA_DIR)
        print(f'Extracted {zip_path.name} into {DATA_DIR}')

ensure_rossmann_dataset()
raw_sales = pd.read_csv(TRAIN_FILE, low_memory=False)
store_meta = pd.read_csv(STORE_FILE, low_memory=False)
print(raw_sales.shape, store_meta.shape)
raw_sales.head()

## Data Validation Checks
Good forecasting starts with defensive validation. We check required columns, malformed dates, duplicate store-date rows, suspicious negative sales, and obvious leakage risks.

In [ ]:
required_sales_cols = {'Store', 'Date', 'Sales', 'Customers', 'Promo', 'Open', 'SchoolHoliday', 'StateHoliday'}
required_store_cols = {'Store'}
missing_sales_cols = required_sales_cols - set(raw_sales.columns)
missing_store_cols = required_store_cols - set(store_meta.columns)
if missing_sales_cols:
    raise ValueError(f'Missing expected sales columns: {sorted(missing_sales_cols)}')
if missing_store_cols:
    raise ValueError(f'Missing expected store columns: {sorted(missing_store_cols)}')

sales = raw_sales.copy()
sales[DATE_COL] = pd.to_datetime(sales[DATE_COL], errors='coerce')
malformed_rows = int(sales[DATE_COL].isna().sum())
duplicate_rows = int(sales.duplicated(subset=['Store', 'Date']).sum())
negative_target_rows = int((sales[TARGET_COL] < 0).sum())
potential_leakage_features = [col for col in sales.columns if col.lower() in {'id'}]

validation_summary = pd.DataFrame([
    {'check': 'malformed_date_rows', 'value': malformed_rows},
    {'check': 'duplicate_store_date_rows', 'value': duplicate_rows},
    {'check': 'negative_sales_rows', 'value': negative_target_rows},
    {'check': 'potential_leakage_feature_count', 'value': len(potential_leakage_features)},
])
validation_summary

## Exploratory Data Analysis
We explore the raw data before any modeling so we understand seasonality, closures, promotions, and distribution shape.

## Target Analysis
Because the target is sales, we inspect both the distribution and the time pattern. This helps us decide whether a seasonal baseline is meaningful and whether outlier handling is needed.

In [ ]:
sales = sales.merge(store_meta, on='Store', how='left')
sales = sales.sort_values(['Store', 'Date']).reset_index(drop=True)
sales = sales[(sales['Open'] != 0) & sales[TARGET_COL].notna()].copy()
series_lengths = sales.groupby(GROUP_COL).size().sort_values(ascending=False)
selected_store = int(series_lengths[series_lengths >= MIN_SERIES_LENGTH].index[0])
series_df = sales.loc[sales[GROUP_COL] == selected_store].copy()
series_df = series_df[[DATE_COL, TARGET_COL, 'Customers', 'Promo', 'SchoolHoliday', 'StateHoliday']].sort_values(DATE_COL)
series_df['StateHoliday'] = series_df['StateHoliday'].astype(str).replace({'0': 'none'})

display(series_df.head())
display(series_df.describe(include='all').T)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
sns.lineplot(data=series_df, x=DATE_COL, y=TARGET_COL, ax=axes[0])
axes[0].set_title(f'Daily sales for Store {selected_store}')
sns.histplot(series_df[TARGET_COL], bins=40, kde=True, ax=axes[1])
axes[1].set_title('Target distribution')
plt.tight_layout()

## Train/Validation/Test Split Strategy
Time order matters, so we split chronologically rather than randomly. The test horizon stays untouched until the very end, and the validation window is used for model ranking and workflow sanity checks.

## Preprocessing Strategy
We keep preprocessing simple and leakage-aware: lag features are built only from past values, calendar variables are derived from the timestamp, categorical fields are one-hot encoded when needed, and scaling is only applied in workflows that benefit from it.

## Feature Engineering
The core engineered features are lags, rolling means, calendar features, and promotion/holiday indicators. These are common and defensible for operational sales forecasting.

In [ ]:
model_df = series_df.copy()
model_df['day_of_week'] = model_df[DATE_COL].dt.dayofweek
model_df['week_of_year'] = model_df[DATE_COL].dt.isocalendar().week.astype(int)
model_df['month'] = model_df[DATE_COL].dt.month
model_df['is_month_start'] = model_df[DATE_COL].dt.is_month_start.astype(int)
model_df['is_month_end'] = model_df[DATE_COL].dt.is_month_end.astype(int)

for lag in [1, 7, 14, 28]:
    model_df[f'sales_lag_{lag}'] = model_df[TARGET_COL].shift(lag)

for window in [7, 14, 28]:
    model_df[f'sales_rollmean_{window}'] = model_df[TARGET_COL].shift(1).rolling(window).mean()

model_df = pd.get_dummies(model_df, columns=['StateHoliday'], drop_first=False)
model_df = model_df.dropna().reset_index(drop=True)

test_df = model_df.tail(FORECAST_HORIZON).copy()
validation_df = model_df.iloc[-2 * FORECAST_HORIZON:-FORECAST_HORIZON].copy()
train_df = model_df.iloc[:-2 * FORECAST_HORIZON].copy()

feature_cols = [col for col in model_df.columns if col not in {DATE_COL, TARGET_COL}]
X_train, y_train = train_df[feature_cols], train_df[TARGET_COL]
X_valid, y_valid = validation_df[feature_cols], validation_df[TARGET_COL]
X_test, y_test = test_df[feature_cols], test_df[TARGET_COL]

train_df[[DATE_COL, TARGET_COL]].tail()

## Baseline Approach
A forecasting notebook should always start with simple baselines. If a complex model cannot beat a naive baseline, the modeling work is not justified.

In [ ]:
def rmse(y_true, y_pred):
    return float(mean_squared_error(y_true, y_pred, squared=False))

def smape(y_true, y_pred):
    denom = np.abs(y_true) + np.abs(y_pred)
    mask = denom != 0
    return float(100 * np.mean(2 * np.abs(y_true[mask] - y_pred[mask]) / denom[mask]))

def evaluate_predictions(name, y_true, y_pred, rows):
    rows.append({
        'model': name,
        'MAE': float(mean_absolute_error(y_true, y_pred)),
        'RMSE': rmse(y_true, y_pred),
        'sMAPE': smape(np.asarray(y_true), np.asarray(y_pred)),
    })

metrics_rows = []
naive_pred = np.repeat(train_df[TARGET_COL].iloc[-1], len(y_valid))
seasonal_naive_pred = validation_df['sales_lag_7'].to_numpy()
evaluate_predictions('Naive last value', y_valid, naive_pred, metrics_rows)
evaluate_predictions('Seasonal naive (lag 7)', y_valid, seasonal_naive_pred, metrics_rows)
pd.DataFrame(metrics_rows).sort_values('RMSE')

## LazyPredict Benchmark
LazyPredict is used only on the lag-feature tabular framing, which is the appropriate way to use it in a forecasting notebook. It is not treated as a native forecasting library.

In [ ]:
lazy_model = LazyRegressor(verbose=0, ignore_warnings=True, custom_metric=None)
lazy_models, lazy_predictions = lazy_model.fit(X_train, X_valid, y_train, y_valid)
lazy_models = lazy_models.reset_index().rename(columns={'index': 'model'})
lazy_models = lazy_models[['model', 'RMSE', 'MAE', 'R-Squared']]
lazy_models.head(10)

## FLAML AutoML Run
FLAML gives a budget-aware AutoML search on the same lag-feature design matrix. This is useful when we want stronger models without manually tuning every learner.

In [ ]:
flaml_automl = AutoML()
flaml_automl.fit(
    X_train=X_train,
    y_train=y_train,
    task='regression',
    time_budget=60,
    metric='rmse',
    seed=SEED,
)
flaml_valid_pred = flaml_automl.predict(X_valid)
evaluate_predictions('FLAML AutoML', y_valid, flaml_valid_pred, metrics_rows)
print('Best FLAML learner:', flaml_automl.best_estimator)

## Additional Best-Library Workflow For The Project Type
StatsForecast is the additional native forecasting workflow. It is appropriate here because daily sales are seasonal, univariate forecasting baselines matter, and statistical models provide a fair benchmark against machine-learning methods.

In [ ]:
stats_train = train_df[[DATE_COL, TARGET_COL]].copy()
stats_train = stats_train.rename(columns={DATE_COL: 'ds', TARGET_COL: 'y'})
stats_train['unique_id'] = 'store_series'

sf = StatsForecast(
    models=[Naive(), SeasonalNaive(season_length=SEASON_LENGTH), AutoARIMA(season_length=SEASON_LENGTH)],
    freq='D',
    n_jobs=-1,
)
sf_forecast = sf.forecast(df=stats_train, h=len(y_valid))
sf_forecast

for model_name in ['Naive', 'SeasonalNaive', 'AutoARIMA']:
    evaluate_predictions(f'StatsForecast {model_name}', y_valid.to_numpy(), sf_forecast[model_name].to_numpy(), metrics_rows)

## Top 3 Model Selection
We select the top 3 approaches based on validation RMSE. That keeps the ranking grounded in actual results instead of preference or hype.

In [ ]:
metrics_df = pd.DataFrame(metrics_rows).drop_duplicates(subset=['model']).sort_values(['RMSE', 'MAE']).reset_index(drop=True)
top3_models = metrics_df.head(3)['model'].tolist()
display(metrics_df)
print('Top 3 models:', top3_models)

## Final Training And Evaluation Of Top 3
Now we retrain the winning workflows on train + validation and evaluate on the untouched test horizon. This is the most honest estimate in the notebook.

In [ ]:
full_train_df = model_df.iloc[:-FORECAST_HORIZON].copy()
X_full_train = full_train_df[feature_cols]
y_full_train = full_train_df[TARGET_COL]
X_test = test_df[feature_cols]
y_test = test_df[TARGET_COL]

final_predictions = {}

if any(name == 'FLAML AutoML' for name in top3_models):
    flaml_final = AutoML()
    flaml_final.fit(X_train=X_full_train, y_train=y_full_train, task='regression', time_budget=60, metric='rmse', seed=SEED)
    final_predictions['FLAML AutoML'] = flaml_final.predict(X_test)

lazy_shortlist = set(top3_models) & set(lazy_models['model'])
for model_name in lazy_shortlist:
    if model_name == 'LightGBMRegressor':
        model = LGBMRegressor(random_state=SEED)
    else:
        model = ExtraTreesRegressor(random_state=SEED, n_estimators=300, n_jobs=-1)
    model.fit(X_full_train, y_full_train)
    final_predictions[model_name] = model.predict(X_test)

if any(name.startswith('StatsForecast') for name in top3_models):
    sf_full_train = full_train_df[[DATE_COL, TARGET_COL]].rename(columns={DATE_COL: 'ds', TARGET_COL: 'y'}).copy()
    sf_full_train['unique_id'] = 'store_series'
    sf_final = StatsForecast(models=[Naive(), SeasonalNaive(season_length=SEASON_LENGTH), AutoARIMA(season_length=SEASON_LENGTH)], freq='D', n_jobs=-1)
    sf_test_forecast = sf_final.forecast(df=sf_full_train, h=len(y_test))
    stats_map = {
        'StatsForecast Naive': 'Naive',
        'StatsForecast SeasonalNaive': 'SeasonalNaive',
        'StatsForecast AutoARIMA': 'AutoARIMA',
    }
    for metric_name, forecast_col in stats_map.items():
        if metric_name in top3_models:
            final_predictions[metric_name] = sf_test_forecast[forecast_col].to_numpy()

final_rows = []
for model_name, preds in final_predictions.items():
    evaluate_predictions(model_name, y_test, preds, final_rows)

final_results = pd.DataFrame(final_rows).sort_values('RMSE').reset_index(drop=True)
final_results

## Error Analysis
Error analysis helps identify when the model struggles, such as promotional spikes, holiday shifts, or unusually weak sales periods.

## Interpretation And Insights
We compare the forecast trajectories directly against the actual values to see whether the winners are capturing weekly seasonality or simply smoothing too aggressively.

In [ ]:
winner = final_results.loc[0, 'model']
winner_pred = final_predictions[winner]
error_df = pd.DataFrame({
    'ds': test_df[DATE_COL].to_numpy(),
    'actual': y_test.to_numpy(),
    'prediction': winner_pred,
})
error_df['abs_error'] = np.abs(error_df['actual'] - error_df['prediction'])
display(error_df.head())
display(error_df.sort_values('abs_error', ascending=False).head(10))

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(error_df['ds'], error_df['actual'], label='Actual', linewidth=2)
ax.plot(error_df['ds'], error_df['prediction'], label=winner, linewidth=2)
ax.set_title(f'Test horizon forecast comparison for Store {selected_store}')
ax.legend()
plt.tight_layout()

## Limitations
- this tutorial models one store series rather than the full multi-store panel
- lag-feature models may miss richer cross-store interactions
- holiday treatment is simplified
- promotions can create spikes that are hard for simple baselines to track

## How To Improve This Project
- move from one store to a multi-series panel forecast
- add store metadata, competition distance, and richer holiday encodings
- tune the validation horizon around the operational planning window
- compare global models against per-store local models

## Production Considerations
- automate daily refreshes with strict date cutoffs
- monitor drift around promotions and holidays
- store forecasts alongside confidence intervals and business overrides
- backtest repeatedly before deployment

## Common Mistakes
- random train/test splits on time-series data
- fitting scalers or encoders on future data
- using LazyPredict as if it were a native forecasting engine
- comparing models only on one metric without reviewing forecast shapes

## Mini Challenge / Exercises
1. Repeat the notebook for the top 5 stores by history length.
2. Replace one-hot holiday encoding with target-aware rolling features.
3. Compare horizon 14, 28, and 42 to see how rankings change.

## Final Summary / Key Takeaways
This notebook demonstrates a complete daily retail forecasting workflow: grounded data download, explicit validation, leakage-aware feature engineering, simple baselines, lag-feature benchmarking with LazyPredict, FLAML AutoML, a native time-series StatsForecast workflow, and a final top-3 comparison on an untouched test horizon.

In [ ]:
artifacts = {
    'project': 'Daily Store Sales Forecasting',
    'selected_store': int(selected_store),
    'validation_results': metrics_df.to_dict(orient='records'),
    'test_results': final_results.to_dict(orient='records'),
    'winner': winner,
}
with open(ARTIFACT_DIR / 'summary.json', 'w', encoding='utf-8') as handle:
    json.dump(artifacts, handle, indent=2)

print('Notebook verification reminders:')
print('- output is .ipynb only')
print('- dataset is downloaded inside the notebook when credentials are available')
print('- LazyPredict is used only on lag features')
print('- FLAML is included')
print('- StatsForecast is the additional forecasting library')

# Daily Store Sales Forecasting

## 1. Title
A learning-first notebook built around the Rossmann Store Sales Kaggle competition data.

## 2. Project Overview
This notebook walks through a reproducible analysis of daily store-level sales. It stays notebook-first, explains every major step, and keeps the workflow suitable for later study or extension into a forecasting project.

## 3. Learning Objectives
- Understand how to acquire competition data from Kaggle inside a notebook.
- Validate the schema before making analytical claims.
- Clean and structure daily store sales data for analysis.
- Explore univariate, bivariate, and multivariate patterns without making unsupported claims.
- Translate observed patterns into cautious business recommendations and follow-up questions.

## 4. Business or Research Problem
Retail teams need a reliable understanding of when and where sales change. This analysis focuses on daily store-level sales behavior so that later forecasting, staffing, promotion planning, and stock decisions can start from a grounded view of the data rather than guesswork.

## 5. Why This Analysis Matters
Poor decisions about promotions, staffing, store operations, or replenishment often start with a weak understanding of sales patterns. A careful exploratory analysis helps separate real signals from noise and reduces the risk of building a forecasting pipeline on bad assumptions.

## 6. Dataset Overview
This project uses the Rossmann Store Sales Kaggle competition data. Based on the verified schema available in this workspace, the notebook expects:

- Sales table columns: `Store`, `DayOfWeek`, `Date`, `Sales`, `Customers`, `Open`, `Promo`, `StateHoliday`, `SchoolHoliday`
- Store metadata columns: `Store`, `StoreType`, `Assortment`, `CompetitionDistance`, `CompetitionOpenSinceMonth`, `CompetitionOpenSinceYear`, `Promo2`, `Promo2SinceWeek`, `Promo2SinceYear`, `PromoInterval`

The sales table contains daily observations, while the store table provides relatively stable attributes for each store.

## 7. Dataset Source and License Notes
Source: Kaggle Rossmann Store Sales competition.

License and usage note: the raw data is governed by Kaggle competition terms. Use it for education, experimentation, and analysis within the allowed terms, and avoid redistributing raw competition files outside those terms.

Important access note: competition datasets often require Kaggle credentials and, in some cases, prior acceptance of the competition rules. The notebook handles that explicitly instead of silently assuming the files already exist.

## 8. Environment Setup

This code cell installs only the packages used in the notebook. Keeping installation logic visible makes the notebook easier to rerun from a clean environment.

In [ ]:
import importlib
import subprocess
import sys

REQUIRED_PACKAGES = {
    "pandas": "pandas",
    "numpy": "numpy",
    "matplotlib": "matplotlib",
    "seaborn": "seaborn",
    "scipy": "scipy",
    "kaggle": "kaggle",
}

def ensure_package(package_name, import_name=None):
    module_name = import_name or package_name
    try:
        importlib.import_module(module_name)
    except Exception:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package_name])

for package_name, import_name in REQUIRED_PACKAGES.items():
    ensure_package(package_name, import_name)

print("Environment setup complete.")

## 9. Imports

These imports are grouped to support data acquisition, validation, cleaning, visualization, and basic statistical testing in one place.

In [ ]:
import json
import os
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display
from scipy.stats import mannwhitneyu

plt.style.use("seaborn-v0_8-whitegrid")
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 200)
np.random.seed(42)

## 10. Configuration / Constants

This cell defines the project paths, the expected Kaggle competition name, and the verified schema used later for validation. The notebook avoids hardcoded absolute machine paths.

In [ ]:
def locate_workspace_root(start_path):
    for candidate in [start_path, *start_path.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "README.md").exists():
            return candidate
    return start_path

WORKSPACE_ROOT = locate_workspace_root(Path.cwd())
PROJECT_DIR = WORKSPACE_ROOT / "Time Series Analysis" / "Daily Store Sales Forecasting"
DATA_DIR = PROJECT_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

KAGGLE_COMPETITION = "rossmann-store-sales"
KAGGLE_ZIP = RAW_DIR / f"{KAGGLE_COMPETITION}.zip"
EXPECTED_TRAIN_FILE = RAW_DIR / "train.csv"
EXPECTED_STORE_FILE = RAW_DIR / "store.csv"
CACHE_DIR = WORKSPACE_ROOT / "data" / "rossmann_store_sales"

EXPECTED_TRAIN_COLUMNS = [
    "Store",
    "DayOfWeek",
    "Date",
    "Sales",
    "Customers",
    "Open",
    "Promo",
    "StateHoliday",
    "SchoolHoliday",
]

EXPECTED_STORE_COLUMNS = [
    "Store",
    "StoreType",
    "Assortment",
    "CompetitionDistance",
    "CompetitionOpenSinceMonth",
    "CompetitionOpenSinceYear",
    "Promo2",
    "Promo2SinceWeek",
    "Promo2SinceYear",
    "PromoInterval",
]

print(json.dumps({
    "workspace_root": str(WORKSPACE_ROOT),
    "project_dir": str(PROJECT_DIR),
    "raw_dir": str(RAW_DIR),
    "cache_dir": str(CACHE_DIR),
    "competition": KAGGLE_COMPETITION,
}, indent=2))

## 11. Dataset Download or Loading

This is the data acquisition block. It attempts a Kaggle competition download first. If that fails because credentials are missing or the competition has not been accepted, the notebook explains the expected file path and schema, then uses the repo-local cache when available so the rest of the notebook can still be studied and executed.

In [ ]:
def kaggle_credentials_available():
    kaggle_json = Path.home() / ".kaggle" / "kaggle.json"
    username = os.getenv("KAGGLE_USERNAME")
    key = os.getenv("KAGGLE_KEY")
    return kaggle_json.exists() or bool(username and key)

def try_kaggle_download():
    if EXPECTED_TRAIN_FILE.exists() and EXPECTED_STORE_FILE.exists():
        print("Raw Kaggle files already exist in the project data folder.")
        return "existing_project_files"

    command = [
        sys.executable,
        "-m",
        "kaggle",
        "competitions",
        "download",
        "-c",
        KAGGLE_COMPETITION,
        "-p",
        str(RAW_DIR),
    ]
    result = subprocess.run(command, capture_output=True, text=True)
    if result.returncode != 0:
        message = (result.stderr or result.stdout or "Unknown Kaggle download failure.").strip()
        raise RuntimeError(message)

    if KAGGLE_ZIP.exists():
        with zipfile.ZipFile(KAGGLE_ZIP, "r") as archive:
            archive.extractall(RAW_DIR)

    if not (EXPECTED_TRAIN_FILE.exists() and EXPECTED_STORE_FILE.exists()):
        raise FileNotFoundError("Kaggle download finished, but expected train.csv and store.csv were not found after extraction.")

    return "kaggle_download"

def copy_workspace_cache():
    cache_train = CACHE_DIR / "data.csv"
    cache_store = CACHE_DIR / "store.csv"
    if cache_train.exists() and cache_store.exists():
        shutil.copy2(cache_train, EXPECTED_TRAIN_FILE)
        shutil.copy2(cache_store, EXPECTED_STORE_FILE)
        return True
    return False

print(f"Kaggle credentials detected: {kaggle_credentials_available()}")
acquisition_mode = None

try:
    acquisition_mode = try_kaggle_download()
except Exception as exc:
    display(Markdown(
        "\n".join([
            "**Kaggle download did not complete.**",
            "",
            f"Reason: `{exc}`",
            "",
            "Expected local files if Kaggle is unavailable:",
            f"- `{EXPECTED_TRAIN_FILE}`",
            f"- `{EXPECTED_STORE_FILE}`",
            "",
            "Expected train schema: " + ", ".join(EXPECTED_TRAIN_COLUMNS),
            "Expected store schema: " + ", ".join(EXPECTED_STORE_COLUMNS),
            "",
            "The notebook will now try the workspace cache so the analysis can still run in this repository.",
        ])
    ))

    if copy_workspace_cache():
        acquisition_mode = "workspace_cache"
    else:
        raise

sales_df = pd.read_csv(EXPECTED_TRAIN_FILE)
store_df = pd.read_csv(EXPECTED_STORE_FILE)

print(f"Acquisition mode: {acquisition_mode}")
print(f"Sales shape: {sales_df.shape}")
print(f"Store shape: {store_df.shape}")
display(sales_df.head())
display(store_df.head())

## 12. Data Validation Checks
Before cleaning or plotting, this block confirms that the dataset actually matches the expected schema and reports missing values, duplicates, and date parsing issues.

## 13. Data Cleaning
The cleaning step keeps the transformations explicit and conservative. We parse dates, remove duplicate keys, merge store metadata, standardize a few categorical values, and create analysis-friendly calendar features. We also keep open-store records separate because many sales analyses become misleading if closed days are mixed in without care.

In [ ]:
missing_train_columns = sorted(set(EXPECTED_TRAIN_COLUMNS) - set(sales_df.columns))
missing_store_columns = sorted(set(EXPECTED_STORE_COLUMNS) - set(store_df.columns))
extra_train_columns = sorted(set(sales_df.columns) - set(EXPECTED_TRAIN_COLUMNS))
extra_store_columns = sorted(set(store_df.columns) - set(EXPECTED_STORE_COLUMNS))

sales_validation = sales_df.copy()
sales_validation["Date"] = pd.to_datetime(sales_validation["Date"], errors="coerce")

validation_report = pd.Series({
    "missing_train_columns": missing_train_columns,
    "missing_store_columns": missing_store_columns,
    "extra_train_columns": extra_train_columns,
    "extra_store_columns": extra_store_columns,
    "sales_rows": int(len(sales_df)),
    "store_rows": int(len(store_df)),
    "unique_stores_in_sales": int(sales_df["Store"].nunique()),
    "unique_stores_in_store_table": int(store_df["Store"].nunique()),
    "invalid_dates": int(sales_validation["Date"].isna().sum()),
    "duplicate_store_date_rows": int(sales_df.duplicated(subset=["Store", "Date"]).sum()),
    "duplicate_store_rows": int(store_df.duplicated(subset=["Store"]).sum()),
}).to_frame(name="value")

sales_missing = sales_df.isna().sum().sort_values(ascending=False).to_frame(name="missing_count")
store_missing = store_df.isna().sum().sort_values(ascending=False).to_frame(name="missing_count")

display(validation_report)
display(sales_missing)
display(store_missing)

sales_clean = sales_df.copy()
store_clean = store_df.copy()

sales_clean["Date"] = pd.to_datetime(sales_clean["Date"], errors="coerce")
sales_clean = sales_clean.dropna(subset=["Date", "Sales"]).drop_duplicates(subset=["Store", "Date"])
store_clean = store_clean.drop_duplicates(subset=["Store"])

analysis_df = sales_clean.merge(store_clean, on="Store", how="left", validate="many_to_one")
analysis_df["StateHoliday"] = analysis_df["StateHoliday"].astype(str).replace({"0": "None"})
analysis_df["PromoInterval"] = analysis_df["PromoInterval"].fillna("None")
analysis_df["Open"] = analysis_df["Open"].fillna(0).astype(int)
analysis_df["Promo"] = analysis_df["Promo"].fillna(0).astype(int)
analysis_df["SchoolHoliday"] = analysis_df["SchoolHoliday"].fillna(0).astype(int)
analysis_df["CompetitionDistanceMissing"] = analysis_df["CompetitionDistance"].isna().astype(int)
analysis_df["CompetitionDistance"] = analysis_df["CompetitionDistance"].fillna(analysis_df["CompetitionDistance"].median())
analysis_df["Year"] = analysis_df["Date"].dt.year
analysis_df["Month"] = analysis_df["Date"].dt.month
analysis_df["MonthName"] = analysis_df["Date"].dt.month_name()
analysis_df["DayName"] = analysis_df["Date"].dt.day_name()
analysis_df["IsWeekend"] = analysis_df["DayOfWeek"].isin([6, 7]).astype(int)

open_sales_df = analysis_df[(analysis_df["Open"] == 1) & (analysis_df["Sales"] > 0)].copy()

cleaning_summary = pd.Series({
    "analysis_rows": int(len(analysis_df)),
    "open_sales_rows": int(len(open_sales_df)),
    "stores_in_analysis": int(analysis_df["Store"].nunique()),
    "date_min": str(analysis_df["Date"].min().date()),
    "date_max": str(analysis_df["Date"].max().date()),
}).to_frame(name="value")

display(cleaning_summary)
display(open_sales_df.head())

## 14. Exploratory Data Analysis
This section builds a broad understanding of scale, date coverage, and variation across stores before moving into more focused questions.

## 15. Univariate Analysis
The next cell looks at individual variable distributions. This is useful for spotting skew, outliers, and category imbalance before comparing variables against each other.

## 19. Visual Analysis
These first visual checks focus on overall scale, record distribution, and high-level store coverage.

In [ ]:
eda_snapshot = pd.Series({
    "row_count": int(len(open_sales_df)),
    "store_count": int(open_sales_df["Store"].nunique()),
    "date_start": str(open_sales_df["Date"].min().date()),
    "date_end": str(open_sales_df["Date"].max().date()),
    "average_daily_sales": float(open_sales_df["Sales"].mean()),
    "median_daily_sales": float(open_sales_df["Sales"].median()),
    "average_customers": float(open_sales_df["Customers"].mean()),
}).to_frame(name="value")

store_level_summary = open_sales_df.groupby("Store").agg(
    average_sales=("Sales", "mean"),
    average_customers=("Customers", "mean"),
    record_count=("Sales", "size"),
).sort_values("average_sales", ascending=False)

display(eda_snapshot)
display(open_sales_df.describe(include="all").transpose())
display(store_level_summary.head(10))

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

axes[0, 0].hist(open_sales_df["Sales"], bins=50, color="steelblue", edgecolor="black")
axes[0, 0].set_title("Distribution of Daily Sales")
axes[0, 0].set_xlabel("Sales")

sns.boxplot(x=open_sales_df["Sales"], ax=axes[0, 1], color="salmon")
axes[0, 1].set_title("Boxplot of Daily Sales")
axes[0, 1].set_xlabel("Sales")

day_counts = open_sales_df["DayName"].value_counts().reindex(["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]).fillna(0)
day_counts.plot(kind="bar", ax=axes[1, 0], color="mediumseagreen")
axes[1, 0].set_title("Open-Day Record Count by Day Name")
axes[1, 0].tick_params(axis="x", rotation=45)

store_type_counts = open_sales_df["StoreType"].value_counts().sort_index()
store_type_counts.plot(kind="bar", ax=axes[1, 1], color="mediumpurple")
axes[1, 1].set_title("Open-Day Record Count by Store Type")

plt.tight_layout()
plt.show()

display(open_sales_df[["Sales", "Customers", "CompetitionDistance"]].describe().transpose())

## 16. Bivariate / Multivariate Analysis
This section compares sales against other observed variables. The goal is to identify useful relationships while remembering that association does not prove causation.

## 17. Feature-Specific Insights
The following cell focuses on store metadata features that are likely to matter for later forecasting and business interpretation, such as store type, assortment, and promotion participation.

## 18. Statistical Checks or Hypothesis Testing Where Relevant
A visual difference is not enough on its own. The next code cell adds non-parametric statistical checks because sales distributions are often skewed and can contain large outliers.

In [ ]:
promo_summary = open_sales_df.groupby("Promo").agg(
    count=("Sales", "size"),
    mean_sales=("Sales", "mean"),
    median_sales=("Sales", "median"),
).reset_index()

holiday_summary = open_sales_df.groupby("SchoolHoliday").agg(
    count=("Sales", "size"),
    mean_sales=("Sales", "mean"),
    median_sales=("Sales", "median"),
).reset_index()

numeric_columns = ["Sales", "Customers", "Promo", "SchoolHoliday", "CompetitionDistance", "DayOfWeek"]
correlation_matrix = open_sales_df[numeric_columns].corr(numeric_only=True)

storetype_summary = open_sales_df.groupby("StoreType").agg(
    mean_sales=("Sales", "mean"),
    median_sales=("Sales", "median"),
    mean_customers=("Customers", "mean"),
    stores=("Store", pd.Series.nunique),
).sort_values("mean_sales", ascending=False)

assortment_summary = open_sales_df.groupby("Assortment").agg(
    mean_sales=("Sales", "mean"),
    median_sales=("Sales", "median"),
    mean_customers=("Customers", "mean"),
).sort_values("mean_sales", ascending=False)

promo2_summary = open_sales_df.groupby("Promo2").agg(
    mean_sales=("Sales", "mean"),
    median_sales=("Sales", "median"),
    mean_customers=("Customers", "mean"),
).reset_index()

def run_mannwhitney_test(frame, group_column, positive_value=1):
    group_a = frame.loc[frame[group_column] == positive_value, "Sales"]
    group_b = frame.loc[frame[group_column] != positive_value, "Sales"]
    if len(group_a) == 0 or len(group_b) == 0:
        return {
            "group_column": group_column,
            "group_a_size": len(group_a),
            "group_b_size": len(group_b),
            "p_value": np.nan,
            "result": "Insufficient data for the comparison.",
        }

    statistic, p_value = mannwhitneyu(group_a, group_b, alternative="two-sided")
    conclusion = "Evidence of a distributional difference" if p_value < 0.05 else "No strong evidence of a distributional difference"
    return {
        "group_column": group_column,
        "group_a_size": int(len(group_a)),
        "group_b_size": int(len(group_b)),
        "u_statistic": float(statistic),
        "p_value": float(p_value),
        "result": conclusion,
        "group_a_median": float(group_a.median()),
        "group_b_median": float(group_b.median()),
    }

promo_test_result = run_mannwhitney_test(open_sales_df, "Promo")
holiday_test_result = run_mannwhitney_test(open_sales_df, "SchoolHoliday")
test_results = pd.DataFrame([promo_test_result, holiday_test_result])

fig, axes = plt.subplots(2, 3, figsize=(20, 10))

sns.barplot(data=promo_summary, x="Promo", y="mean_sales", ax=axes[0, 0], palette="Blues")
axes[0, 0].set_title("Average Sales by Promo Status")

sns.barplot(data=holiday_summary, x="SchoolHoliday", y="mean_sales", ax=axes[0, 1], palette="Greens")
axes[0, 1].set_title("Average Sales by School Holiday Status")

sns.heatmap(correlation_matrix, annot=True, cmap="coolwarm", fmt=".2f", ax=axes[0, 2])
axes[0, 2].set_title("Correlation Heatmap")

sns.barplot(data=storetype_summary.reset_index(), x="StoreType", y="mean_sales", ax=axes[1, 0], palette="viridis")
axes[1, 0].set_title("Average Sales by Store Type")

sns.barplot(data=assortment_summary.reset_index(), x="Assortment", y="mean_sales", ax=axes[1, 1], palette="magma")
axes[1, 1].set_title("Average Sales by Assortment")

sns.barplot(data=promo2_summary, x="Promo2", y="mean_sales", ax=axes[1, 2], palette="crest")
axes[1, 2].set_title("Average Sales by Promo2 Participation")

plt.tight_layout()
plt.show()

display(promo_summary)
display(holiday_summary)
display(correlation_matrix)
display(storetype_summary)
display(assortment_summary)
display(promo2_summary)
display(test_results)

## 20. Key Findings
The notebook should not claim findings before the code runs. The next cell creates a concise, evidence-based summary directly from the computed tables so the interpretation stays tied to the actual data in memory.

In [ ]:
daily_sales = open_sales_df.groupby("Date", as_index=False)["Sales"].sum()
weekday_sales = open_sales_df.groupby("DayName", as_index=False)["Sales"].mean()
weekday_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
weekday_sales["DayName"] = pd.Categorical(weekday_sales["DayName"], categories=weekday_order, ordered=True)
weekday_sales = weekday_sales.sort_values("DayName")

monthly_sales = open_sales_df.groupby(["Year", "Month"], as_index=False)["Sales"].mean()
customer_sample = open_sales_df.sample(min(len(open_sales_df), 15000), random_state=42)

fig, axes = plt.subplots(2, 2, figsize=(18, 12))

axes[0, 0].plot(daily_sales["Date"], daily_sales["Sales"], color="navy")
axes[0, 0].set_title("Total Daily Sales Over Time")
axes[0, 0].set_xlabel("Date")
axes[0, 0].set_ylabel("Total Sales")

sns.barplot(data=weekday_sales, x="DayName", y="Sales", ax=axes[0, 1], palette="Set2")
axes[0, 1].set_title("Average Sales by Day of Week")
axes[0, 1].tick_params(axis="x", rotation=45)

sns.lineplot(data=monthly_sales, x="Month", y="Sales", hue="Year", marker="o", ax=axes[1, 0])
axes[1, 0].set_title("Average Sales by Month and Year")
axes[1, 0].set_ylabel("Average Sales")

sns.scatterplot(data=customer_sample, x="Customers", y="Sales", hue="Promo", alpha=0.35, ax=axes[1, 1])
axes[1, 1].set_title("Customers vs Sales")

plt.tight_layout()
plt.show()

promo_mean_map = promo_summary.set_index("Promo")["mean_sales"].to_dict()
promo_lift_pct = np.nan
if 0 in promo_mean_map and 1 in promo_mean_map and promo_mean_map[0] != 0:
    promo_lift_pct = ((promo_mean_map[1] - promo_mean_map[0]) / promo_mean_map[0]) * 100

top_store_type = storetype_summary.reset_index().iloc[0]
top_assortment = assortment_summary.reset_index().iloc[0]
top_weekday = weekday_sales.loc[weekday_sales["Sales"].idxmax()]

findings = [
    f"The cleaned open-store analysis covers {open_sales_df['Store'].nunique():,} stores from {open_sales_df['Date'].min().date()} to {open_sales_df['Date'].max().date()}.",
    f"Average sales on promo days are {promo_lift_pct:.2f}% {'higher' if promo_lift_pct >= 0 else 'lower'} than non-promo days in this sample." if not np.isnan(promo_lift_pct) else "Promo lift could not be computed from the current data split.",
    f"The highest average sales by weekday appear on {top_weekday['DayName']}.",
    f"Store type {top_store_type['StoreType']} has the highest observed average sales among store types in this dataset.",
    f"Assortment category {top_assortment['Assortment']} ranks highest by average observed sales in this dataset.",
    f"Promo statistical check: {promo_test_result['result']} (p-value={promo_test_result['p_value']:.6f})." if not np.isnan(promo_test_result['p_value']) else "Promo statistical check was not available.",
    f"School holiday statistical check: {holiday_test_result['result']} (p-value={holiday_test_result['p_value']:.6f})." if not np.isnan(holiday_test_result['p_value']) else "School holiday statistical check was not available.",
]

for item in findings:
    print(f"- {item}")

## 21. Limitations
- This is observational sales data, so differences across promotions or holidays should not be treated as causal proof.
- `Customers` is useful for analysis, but it is a leakage-risk feature for future forecasting because it is observed after the selling day happens.
- Competition data may differ from real operational data quality, especially if future planning inputs are missing.
- The notebook uses the verified schema available in this repository; if Kaggle changes packaging or file names, the loading cell may need small adjustments.

## 22. Recommendations / Next Steps
- Use the strongest observed group patterns here as candidates for later forecasting features, not as final business rules.
- Build a time-aware forecasting notebook next, and avoid using `Customers` as an input feature for future prediction.
- Review stores with unusually high or low average sales separately to check whether they reflect store type, assortment, competition distance, or data quality issues.
- How to extend this analysis: add store-level trend decomposition, evaluate holiday calendars more carefully, and compare local store segments instead of relying only on global averages.

## 23. Common Mistakes
- Treating closed days and open days as equivalent without checking how zero sales behave.
- Making claims about promotion impact without checking both descriptive summaries and a statistical test.
- Using `Customers` in a future forecasting model as if it were known in advance.
- Ignoring schema validation and assuming the downloaded files always match the expected Kaggle packaging.
- Reading correlations as causal effects.

## 24. Mini Challenge / Exercises
1. Re-run the bivariate and statistical comparison for `StateHoliday` instead of `SchoolHoliday`.
2. Create a store-level ranking of the most promotion-sensitive stores and inspect whether the pattern is stable across store types.
3. Replace the overall time plots with one selected store and one selected store type to compare local versus global behavior.
4. Add a separate analysis for stores with missing `CompetitionDistance` before and after imputation.

## 25. Final Summary / Key Takeaways
This notebook keeps the workflow direct and reproducible: it acquires or loads the data, validates the schema, cleans the records, and builds a cautious analytical picture of daily store sales. The most important habit is not any single chart or test. It is the sequence: validate first, analyze second, and only then decide what deserves to become a forecasting feature or business action.

After running the notebook top to bottom, revisit the generated findings cell and the plots together. That combination gives a much stronger basis for next-step forecasting work than relying on one metric or one visual alone.